In [1]:
import pathlib
import os
from typing import List

from datasets import load_dataset, load_from_disk
import evaluate
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score


def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    print(f"Loading pretrained model from {path}")
    checkpoint = torch.load(path, map_location=device)

    gptconf = GPTConfig(**checkpoint['model_args'])
    pretrained_model = GPT(gptconf)
    state_dict = checkpoint['model']

    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model_dict = pretrained_model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items()
                           if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_state_dict)
    pretrained_model.load_state_dict(model_dict)
    pretrained_model.to(device)

    return pretrained_model


# Define variables directly here instead of using argparse
args = {
    'vocab': pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-vocab.json"),  # Example for IPA
    'merges': pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-merges.txt"),  # Example for IPA
    'model': pathlib.Path("/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_50k/ckpt.pt"),  # IPA Model
    'task': "sst2",  # Example task
    'epochs': 3,
    'eval_interval': 0.01,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 8,
    'hf_cache_dir': pathlib.Path('cache'),
    'dataset': 'iggy12345/glue-sst2-ipa',  # Dataset path
    'from_disk': False,
    'no_subset': False,
    'device': 'cuda',
    'no_progress_bar': False
}

# ---- Models and Tokenizers ----
models_and_tokenizers = [               #/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k
    {"model_type": "ipa", "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-merges.txt")},
    {"model_type": "normal", "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation-merges.txt")},
    {"model_type": "prebuilt", "model_path": "/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_medium/ckpt.pt",
     "tokenizer_paths": None}
]

# Create a main directory for all model outputs
main_output_dir = pathlib.Path("./training_outputs_sst2")
if not os.path.exists(main_output_dir):
    os.makedirs(main_output_dir)

# Loop through the models and tokenizers
for config in models_and_tokenizers:
    model_type = config["model_type"]
    model_path = config["model_path"]
    tokenizer_paths = config["tokenizer_paths"]

    # Dynamically set the output directory based on the model type
    output_dir = main_output_dir / f"output_{model_type}"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # ---- Load Tokenizer ----
    if model_type == "ipa" or model_type == "normal":
        vocab_path, merges_path = tokenizer_paths
        tokenizer = load_tokenizer(vocab_path, merges_path)
    elif model_type == "prebuilt":
        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token

    # ---- Load model ----
    base_model = load_pretrained_model(model_path, args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model).to(args['device'])

    # ---- Load dataset ----
    dataset = load_dataset(args['dataset'], cache_dir=str(args['hf_cache_dir']))

    def flatten_multi_features(examples, features: List[str]) -> List[str]:
        separator = f'\n\n{eod_token}\n\n'
        return [separator.join(example) for example in zip(*[examples[f] for f in features])]

    # ---- Preprocessing ----
    def preprocess_function(examples):
        if 'premise' in examples:
            feature = flatten_multi_features(examples, ['premise', 'hypothesis'])
        elif 'question' in examples:
            if 'sentence' in examples:
                feature = flatten_multi_features(examples, ['question', 'sentence'])
            else:
                feature = flatten_multi_features(examples, ['question', 'hypothesis'])
        elif 'sentence1' in examples:
            feature = flatten_multi_features(examples, ['sentence1', 'sentence2'])
        elif 'question1' in examples:
            feature = flatten_multi_features(examples, ['question1', 'question2'])
        else:
            feature = examples['sentence']

        return tokenizer(feature, truncation=True, max_length=args['context_size'])

    encoded_dataset = dataset.map(preprocess_function, batched=True)

    # ---- Data collator ----
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # ---- Metrics ----
    metric = evaluate.load("glue", args['task'])

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = torch.from_numpy(logits).argmax(dim=-1)
        # return metric.compute(predictions=predictions, references=labels)
        hf_metrics = metric.compute(predictions=predictions, references=labels)
        hf_metrics["precision"] = precision_score(labels, predictions)
        hf_metrics["recall"] = recall_score(labels, predictions)
        hf_metrics["f1"] = f1_score(labels, predictions)
        return hf_metrics

    # ---- Training arguments ----
    training_args = TrainingArguments(
        output_dir=str(output_dir),  # Dynamically set output directory
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",  # Save every 'save_steps'
        save_steps=500,  # Save every 100 steps
        save_total_limit=1,  # Keep only 1 checkpoint
        metric_for_best_model="accuracy",
        load_best_model_at_end=True,  # Automatically load the best model after training
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=500,  # Log every 100 steps
        logging_dir='./logs',  # Log directory to store logs
        disable_tqdm=False,  # Show progress bar in notebook
        warmup_ratio=0.3,  # Try a warmup ratio
        save_safetensors=False  # Disable safetensors saving
    )

    # ---- Trainer ----
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # ---- Train ----
    trainer.train(resume_from_checkpoint=False)  
    results = trainer.evaluate(encoded_dataset["validation"])
    print(f"Evaluation results for {model_type}: {results}")


/users/PAS2836/krishnakb/ondemand/krishna_proj/cleanenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pretrained model from /fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt
number of parameters: 353.24M


/tmp/slurmtmp.1402334/ipykernel_881622/1393725084.py:172: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: orugantikoundinya7 (orugantikoundinya7-ohio-state-buckeyes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,0.566400,0.349885,0.868119,0.883450,0.853604,0.868270
1000,0.385500,0.296418,0.896789,0.907834,0.887387,0.897494
1500,0.381900,0.724412,0.872706,0.815939,0.968468,0.885685
2000,0.384600,0.376860,0.881881,0.840319,0.948198,0.891005
2500,0.355000,0.801098,0.818807,0.746552,0.975225,0.845703
3000,0.347700,0.421687,0.887615,0.911905,0.862613,0.886574
3500,0.352600,0.390982,0.873853,0.897619,0.849099,0.872685
4000,0.350200,0.493599,0.853211,0.802682,0.943694,0.867495
4500,0.347000,0.384590,0.884174,0.881960,0.891892,0.886898
5000,0.332800,0.614606,0.873853,0.832669,0.941441,0.883721


Evaluation results for ipa: {'eval_loss': 0.4450751543045044, 'eval_accuracy': 0.9071100917431193, 'eval_precision': 0.9115646258503401, 'eval_recall': 0.9054054054054054, 'eval_f1': 0.9084745762711864, 'eval_runtime': 3.6341, 'eval_samples_per_second': 239.951, 'eval_steps_per_second': 29.994, 'epoch': 3.0}
Loading pretrained model from /fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt
number of parameters: 353.24M


Map: 100%|██████████| 1821/1821 [00:00<00:00, 27550.01 examples/s]
/tmp/slurmtmp.1402334/ipykernel_881622/1393725084.py:172: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,0.498800,0.348344,0.879587,0.898824,0.860360,0.879171
1000,0.404700,0.303794,0.892202,0.944162,0.837838,0.887828
1500,0.369600,0.518816,0.891055,0.872068,0.921171,0.895947
2000,0.387700,0.348553,0.888761,0.854806,0.941441,0.896034
2500,0.368900,0.471690,0.875000,0.823985,0.959459,0.886576
3000,0.330300,0.488267,0.884174,0.936387,0.828829,0.879331
3500,0.348800,0.388601,0.885321,0.930000,0.837838,0.881517
4000,0.343400,0.357176,0.905963,0.895197,0.923423,0.909091
4500,0.355200,0.407052,0.895642,0.893096,0.903153,0.898096
5000,0.331000,0.568616,0.873853,0.830040,0.945946,0.884211


Evaluation results for normal: {'eval_loss': 0.3571757674217224, 'eval_accuracy': 0.9059633027522935, 'eval_precision': 0.8951965065502183, 'eval_recall': 0.9234234234234234, 'eval_f1': 0.9090909090909091, 'eval_runtime': 3.608, 'eval_samples_per_second': 241.685, 'eval_steps_per_second': 30.211, 'epoch': 3.0}
Loading pretrained model from /fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_medium/ckpt.pt


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory